In [1]:
import pickle
import random

import torch

# Fix the kernel dying?
#import os
#os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

#from dollar_B import *

In [2]:
tensor_dict_path = "C:\\Users\\kdmen\\Repos\\pers-gest-cls\\dataset\\meta-learning-sup-que-ds\\maml_tensor_dict.pkl"

with open(tensor_dict_path, "rb") as f:
    tensor_dict = pickle.load(f)

In [3]:
def ensure_channel_first(x: torch.Tensor) -> torch.Tensor:
    """Helper from eval_knn_proto.py to ensure (N, C, T) shape."""
    if x is None or x.dim() != 3:
        return x
    # If the last dimension matches known channel counts, swap it
    if x.shape[-1] in [16, 72]:
        return x.permute(0, 2, 1)
    return x

In [4]:
import os
import pickle
import random
import torch
from typing import Optional, List, Tuple

# =============================================================================
# 1. Episode Builder (Replaces your old build_episode)
# =============================================================================

def build_episode_dollar_B(
    user_data: dict, 
    k_shot: int, 
    n_way: int, 
    rng: random.Random,
    all_reps: List[int],
) -> Tuple[dict, dict]:
    """
    Extracts k_shot templates and the remaining query samples for n_way classes.
    Fixed split: first k_shot trials are support, the rest are query.
    """
    available_classes = [g for g in all_reps if g in user_data]
    chosen_classes = available_classes[:n_way]
    
    sup_emg, sup_imu, sup_labels = [], [], []
    qry_emg, qry_imu, qry_labels = [], [], []
    
    for c_idx, g in enumerate(chosen_classes):
        # NOTE: BY DEFAULT, data loads in as (N_trials, T, channels)
        emg_tensor = ensure_channel_first(user_data[g]["emg"])  # (N_trials, C_emg, T)
        imu_tensor = ensure_channel_first(user_data[g]["imu"])  # (N_trials, C_imu, T)
        
        # We assume trials are ordered. Take the first k_shot as templates.
        sup_emg.append(emg_tensor[:k_shot])
        sup_imu.append(imu_tensor[:k_shot])
        sup_labels.append(torch.full((k_shot,), c_idx, dtype=torch.long))
        
        # Take the rest as queries
        qry_emg.append(emg_tensor[k_shot:])
        qry_imu.append(imu_tensor[k_shot:])
        num_queries = emg_tensor.shape[0] - k_shot
        qry_labels.append(torch.full((num_queries,), c_idx, dtype=torch.long))
        
    support = {
        "emg": torch.cat(sup_emg, dim=0),
        "imu": torch.cat(sup_imu, dim=0),
        "labels": torch.cat(sup_labels, dim=0)
    }
    query = {
        "emg": torch.cat(qry_emg, dim=0),
        "imu": torch.cat(qry_imu, dim=0),
        "labels": torch.cat(qry_labels, dim=0)
    }
    return support, query


# =============================================================================
# 2. The Exact $B Classifier 
# =============================================================================

def exact_dollar_B_classify(
    sup_emg: torch.Tensor,   
    sup_imu: torch.Tensor, 
    sup_labels: torch.Tensor,   
    qry_emg: torch.Tensor,   
    qry_imu: torch.Tensor, 
    n_components: int = 50,
) -> torch.Tensor:
    """
    1. Early fusion (concatenate EMG & IMU vertically).
    2. Per-TEMPLATE (per-sample) PCA.
    3. 1-Nearest Neighbor based on L1 distance.
    """
    # 1. Early Concatenation of Modalities (Shared Space)
    sup = torch.cat([sup_emg, sup_imu], dim=1)  # (N_sup, C_emg + C_imu, T)
    qry = torch.cat([qry_emg, qry_imu], dim=1)  # (N_qry, C_emg + C_imu, T)

    N_sup, C, T = sup.shape
    N_qry = qry.shape[0]
    n_pc = min(n_components, C - 1)
    
    # Distance from each query to each INDIVIDUAL support template
    all_dists = torch.zeros(N_qry, N_sup, device=qry.device)

    # 2. Per-Template Processing
    for i in range(N_sup):
        template_ts = sup[i]                     # (C, T)
        mean_c = template_ts.mean(dim=1, keepdim=True) # (C, 1)
        x_centered = template_ts - mean_c        # (C, T)
        
        # Fit PCA on this exact template
        cov = (x_centered @ x_centered.t()) / (T - 1)
        
        try:
            _, vecs = torch.linalg.eigh(cov)
            U = vecs[:, -n_pc:].flip(dims=[1])   # (C, n_pc)
        except Exception:
            U = torch.eye(C, device=sup.device)[:, :n_pc]
            
        # Transform template to its own latent space and flatten
        proj_t = (U.t() @ x_centered).flatten()  # (n_pc * T,)
        
        # 3. Project all queries into THIS template's latent space
        for q in range(N_qry):
            xq_centered = qry[q] - mean_c        # (C, T)
            proj_q = (U.t() @ xq_centered).flatten() # (n_pc * T,)
            
            # L1 Distance
            all_dists[q, i] = (proj_q - proj_t).abs().sum()

    # 1-NN: find the single closest template index
    best_template_idx = all_dists.argmin(dim=1)  # (N_qry,)
    return sup_labels[best_template_idx]         # (N_qry,)


# =============================================================================
# 3. Evaluator Pipeline
# =============================================================================

def run_dollar_B_evaluation(
    tensor_dict: dict,
    k_shot: int,
    n_way: int = 10,
    n_components: int = 50,
    seed: int = 42
):
    print(f"\n{'='*65}")
    print(f"  EXACT $B EVAL: {k_shot}-shot  {n_way}-way")
    print(f"  n_components : {n_components} (EMG+IMU Early Fusion)")
    print(f"{'='*65}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    rng = random.Random(seed)
    
    # Extract list of all possible reps/gestures across the dataset
    # We grab this from the first user's dictionary keys
    first_pid = list(tensor_dict.keys())[0]
    all_reps = list(tensor_dict[first_pid].keys()) 
    
    accs = []
    
    for pid, user_data in tensor_dict.items():
        available = [g for g in all_reps if g in user_data]
        if len(available) < n_way: 
            continue # Skip if user doesn't have enough classes
            
        # Build episode for this user
        support, query = build_episode_dollar_B(
            user_data=user_data, 
            k_shot=k_shot, 
            n_way=n_way, 
            rng=rng, 
            all_reps=all_reps
        )

        sup_emg = support["emg"].to(device)
        qry_emg = query["emg"].to(device)
        sup_imu = support["imu"].to(device)
        qry_imu = query["imu"].to(device)
        sup_labels = support["labels"].to(device)
        qry_labels = query["labels"].to(device)
        
        # Skip if they don't have enough shots
        if sup_emg.size(0) < k_shot * n_way:
            continue

        # Run Prediction
        preds = exact_dollar_B_classify(
            sup_emg=sup_emg, 
            sup_imu=sup_imu, 
            sup_labels=sup_labels,
            qry_emg=qry_emg, 
            qry_imu=qry_imu,
            n_components=n_components
        )
        
        # Calculate Accuracy
        acc = (preds == qry_labels).float().mean().item()
        accs.append(acc)
        
        print(f"  PID {str(pid):>4} | exact_$B = {acc*100:>5.1f}%  [sup={sup_emg.size(0)}, qry={qry_emg.size(0)}]")

    if len(accs) > 0:
        mean_acc = torch.tensor(accs).mean().item()
        std_acc  = torch.tensor(accs).std().item()

        print(f"\n  {'Method':<30}  {'Mean':>8}  {'Std':>7}  {'Users':>6}")
        print(f"  {'─'*30}  {'─'*8}  {'─'*7}  {'─'*6}")
        print(f"  {'Exact $B ($B-faithful)':<30}  {mean_acc*100:>7.2f}%  {std_acc*100:>6.2f}%  {len(accs):>6}")
    else:
        print("No valid users found to evaluate.")


In [5]:
# Fix potential kernel crashing on Windows
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# Load your dataset
tensor_dict_path = r"C:\Users\kdmen\Repos\pers-gest-cls\dataset\meta-learning-sup-que-ds\maml_tensor_dict.pkl"

print("Loading data...")
with open(tensor_dict_path, "rb") as f:
    tensor_dict = pickle.load(f)
print("Data loaded!")

# Run for 1-shot (and 3-shot / 5-shot if you like)
for shots in [1, 3, 5]:
    run_dollar_B_evaluation(
        tensor_dict=tensor_dict,
        k_shot=shots,
        n_way=10,            # 10-class problem
        n_components=50,     # Enforce the shared 50 PCs
        seed=42
    )

Loading data...
Data loaded!

  EXACT $B EVAL: 1-shot  10-way
  n_components : 50 (EMG+IMU Early Fusion)
  PID P102 | exact_$B =  12.2%  [sup=10, qry=90]
  PID P103 | exact_$B =  12.2%  [sup=10, qry=90]
  PID P104 | exact_$B =  10.0%  [sup=10, qry=90]
  PID P105 | exact_$B =  12.2%  [sup=10, qry=90]
  PID P106 | exact_$B =   5.6%  [sup=10, qry=90]
  PID P107 | exact_$B =  14.4%  [sup=10, qry=90]
  PID P108 | exact_$B =   6.7%  [sup=10, qry=90]
  PID P109 | exact_$B =  12.2%  [sup=10, qry=90]
  PID P110 | exact_$B =  11.1%  [sup=10, qry=90]
  PID P111 | exact_$B =  10.0%  [sup=10, qry=90]
  PID P112 | exact_$B =  11.1%  [sup=10, qry=90]
  PID P114 | exact_$B =  10.0%  [sup=10, qry=90]
  PID P115 | exact_$B =   6.7%  [sup=10, qry=90]
  PID P116 | exact_$B =   7.8%  [sup=10, qry=90]
  PID P118 | exact_$B =  10.0%  [sup=10, qry=90]
  PID P119 | exact_$B =   8.9%  [sup=10, qry=90]
  PID P121 | exact_$B =  13.3%  [sup=10, qry=90]
  PID P122 | exact_$B =  12.2%  [sup=10, qry=90]
  PID P123 | 